# Preparación de las selecciones del Mundial 2026

En este notebook se construye el dataset de las selecciones clasificadas al Mundial 2026 utilizando los rankings FIFA, ratings Elo y métricas de forma reciente.

In [1]:
import pandas as pd
import numpy as np

In [2]:
fifa = pd.read_csv("../data/processed/fifa_normalized.csv")
elo = pd.read_csv("../data/processed/elo_normalized.csv")
dataset = pd.read_csv("../data/processed/dataset_mundial_final.csv")

In [6]:
fifa["rank_date"] = pd.to_datetime(
    fifa["rank_date"]
)

elo["date"] = pd.to_datetime(
    elo["date"]
)

In [31]:
fifa_actual = (
    fifa
    .sort_values("rank_date")
    .groupby("country_full")
    .tail(1)
)

fifa_actual.head()

,rank_date,country_full,rank,total_points
6715,2002-11-20,Yugoslavia,16.0,687.00
15426,2006-07-12,Serbia and Montenegro,36.0,724.00
26691,2011-02-02,Netherlands Antilles,150.0,107.00
53363,2023-07-20,Eritrea,195.0,855.56
57126,2025-12-19,Congo,134.0,1105.96


In [8]:
selecciones_mundial = [
    "Mexico",
    "South Africa",
    "South Korea",
    "Czech Republic",

    "Canada",
    "Bosnia and Herzegovina",
    "Qatar",
    "Switzerland",

    "Brazil",
    "Morocco",
    "Haiti",
    "Scotland",

    "United States",
    "Paraguay",
    "Australia",
    "Turkey",

    "Germany",
    "Curacao",
    "Ivory Coast",
    "Ecuador",

    "Netherlands",
    "Japan",
    "Sweden",
    "Tunisia",

    "Belgium",
    "Egypt",
    "Iran",
    "New Zealand",

    "Spain",
    "Cape Verde",
    "Saudi Arabia",
    "Uruguay",

    "France",
    "Senegal",
    "Iraq",
    "Norway",

    "Argentina",
    "Algeria",
    "Austria",
    "Jordan",

    "Portugal",
    "DR Congo",
    "Uzbekistan",
    "Colombia",

    "England",
    "Croatia",
    "Ghana",
    "Panama"
]

In [9]:
selecciones_df = pd.DataFrame({
    "team": selecciones_mundial
})

selecciones_df.shape

(48, 1)

In [11]:
fifa_actual = fifa_actual[
    fifa_actual["country_full"].isin(selecciones_mundial)
]

fifa_actual.shape

(39, 4)

In [12]:
fifa_actual.sort_values("rank")

,rank_date,country_full,rank,total_points
56993,2025-12-19,Spain,1.0,1877.18
56994,2025-12-19,Argentina,2.0,1873.33
56995,2025-12-19,France,3.0,1870.00
56996,2025-12-19,England,4.0,1834.12
56997,2025-12-19,Brazil,5.0,1760.46
56998,2025-12-19,Portugal,6.0,1760.38
56999,2025-12-19,Netherlands,7.0,1756.27
57000,2025-12-19,Belgium,8.0,1730.71
57001,2025-12-19,Germany,9.0,1724.15
57002,2025-12-19,Croatia,10.0,1716.88


In [13]:
len(fifa_actual)

39

In [14]:
faltan_fifa = set(selecciones_mundial) - set(fifa_actual["country_full"])

faltan_fifa

{'Cape Verde',
 'Curacao',
 'Czech Republic',
 'DR Congo',
 'Iran',
 'Ivory Coast',
 'South Korea',
 'Turkey',
 'United States'}

In [16]:
for texto in [
    "Korea",
    "Turkey",
    "Iran",
    "Congo",
    "United",
    "Verde",
    "Coast",
    "Curacao",
    "Czech"
]:
    print("\n", texto)
    print(
        fifa["country_full"]
        .loc[
            fifa["country_full"].str.contains(
                texto,
                case=False,
                na=False
            )
        ]
        .drop_duplicates()
        .tolist()
    )


 Korea
['Korea Republic', 'Korea DPR']

 Turkey
[]

 Iran
['IR Iran']

 Congo
['Congo DR', 'Congo']

 United
['United Arab Emirates']

 Verde
['Cabo Verde']

 Coast
[]

 Curacao
[]

 Czech
['Czechia']


In [17]:
for texto in [
    "United",
    "America",
    "USA",
    "Cape",
    "Verde",
    "Cote",
    "Ivoire",
    "Curacao",
    "Turkey",
    "Turkiye"
]:
    print("\n", texto)

    print(
        fifa["country_full"]
        .loc[
            fifa["country_full"].str.contains(
                texto,
                case=False,
                na=False
            )
        ]
        .drop_duplicates()
        .tolist()
    )


 United
['United Arab Emirates']

 America
['American Samoa']

 USA
['USA']

 Cape
[]

 Verde
['Cabo Verde']

 Cote
[]

 Ivoire
["Côte d'Ivoire"]

 Curacao
[]

 Turkey
[]

 Turkiye
[]


In [18]:
fifa_actual[
    fifa_actual["country_full"].str.contains(
        "Tur",
        case=False,
        na=False
    )
]

,rank_date,country_full,rank,total_points


In [19]:
fifa_actual[
    fifa_actual["country_full"].str.contains(
        "Cur",
        case=False,
        na=False
    )
]

,rank_date,country_full,rank,total_points


In [20]:
fifa_actual[
    fifa_actual["country_full"].str.contains(
        "Bos",
        case=False,
        na=False
    )
]

,rank_date,country_full,rank,total_points
57063,2025-12-19,Bosnia and Herzegovina,71.0,1362.37


In [21]:
fifa[
    fifa["country_full"].str.contains(
        "Tür",
        case=False,
        na=False
    )
]["country_full"].drop_duplicates()

93    Türkiye
Name: country_full, dtype: str

In [22]:
fifa[
    fifa["country_full"].str.contains(
        "Cura",
        case=False,
        na=False
    )
]["country_full"].drop_duplicates()

26463    Curaçao
Name: country_full, dtype: str

In [33]:
equivalencias_fifa = {
    "South Korea": "Korea Republic",
    "Iran": "IR Iran",
    "DR Congo": "Congo DR",
    "Czech Republic": "Czechia",
    "Cape Verde": "Cabo Verde",
    "Ivory Coast": "Côte d'Ivoire",
    "United States": "USA",
    "Turkey": "Türkiye",
    "Curacao": "Curaçao"
}

In [34]:
selecciones_df["country_fifa"] = (
    selecciones_df["team"]
    .replace(equivalencias_fifa)
)

In [35]:
fifa_mundial = (
    selecciones_df.merge(
        fifa_actual,
        left_on="country_fifa",
        right_on="country_full",
        how="left"
    )
)

In [43]:
fifa_mundial.shape

(48, 6)

In [42]:
fifa_mundial[
    ["team", "rank", "total_points"]
].head()

,team,rank,total_points
0,Mexico,15.0,1675.75
1,South Africa,61.0,1426.73
2,South Korea,22.0,1599.45
3,Czech Republic,44.0,1487.00
4,Canada,27.0,1559.15


In [37]:
fifa_mundial["rank"].isna().sum()

np.int64(0)

In [39]:
fifa_mundial[
    fifa_mundial["rank"].isna()
][
    ["team", "country_fifa"]
]

,team,country_fifa


In [41]:
len(fifa_actual)

214

In [45]:
fifa_mundial = fifa_mundial[
    [
        "team",
        "rank",
        "total_points"
    ]
]

fifa_mundial.head(20)

,team,rank,total_points
0,Mexico,15.0,1675.75
1,South Africa,61.0,1426.73
2,South Korea,22.0,1599.45
3,Czech Republic,44.0,1487.00
4,Canada,27.0,1559.15
5,Bosnia and Herzegovina,71.0,1362.37
6,Qatar,54.0,1454.96
7,Switzerland,17.0,1654.69
8,Brazil,5.0,1760.46
9,Morocco,11.0,1716.34


In [46]:
elo.columns

Index(['date', 'team', 'rating'], dtype='str')

In [47]:
elo_actual = (
    elo
    .sort_values("date")
    .groupby("team")
    .tail(1)
)

len(elo_actual)

248

In [48]:
faltan_elo = (
    set(selecciones_mundial)
    - set(elo_actual["team"])
)

faltan_elo

{'Curacao', 'Czech Republic', 'DR Congo'}

In [49]:
elo_actual[
    elo_actual["team"].str.contains(
        "Czech|Cura|Congo",
        case=False,
        na=False
    )
][["team", "rating"]]

,team,rating
4986,Congo,1206.0
4866,Czechia,1731.0
4887,Democratic Republic of Congo,1616.0
4915,Curaçao,1467.0


In [50]:
equivalencias_elo = {
    "Curacao": "Curaçao",
    "Czech Republic": "Czechia",
    "DR Congo": "Democratic Republic of Congo"
}

In [51]:
selecciones_df["team_elo"] = (
    selecciones_df["team"]
    .replace(equivalencias_elo)
)

In [52]:
elo_mundial = (
    selecciones_df.merge(
        elo_actual,
        left_on="team_elo",
        right_on="team",
        how="left"
    )
)

In [53]:
elo_mundial["rating"].isna().sum()

np.int64(0)

In [54]:
elo_mundial = elo_mundial[
    [
        "team_x",
        "rating"
    ]
]

elo_mundial.columns = [
    "team",
    "elo"
]

elo_mundial.head()

,team,elo
0,Mexico,1835.0
1,South Africa,1531.0
2,South Korea,1784.0
3,Czech Republic,1731.0
4,Canada,1802.0


In [55]:
simulacion_df = (
    fifa_mundial.merge(
        elo_mundial,
        on="team",
        how="inner"
    )
)

simulacion_df.head()

,team,rank,total_points,elo
0,Mexico,15.0,1675.75,1835.0
1,South Africa,61.0,1426.73,1531.0
2,South Korea,22.0,1599.45,1784.0
3,Czech Republic,44.0,1487.00,1731.0
4,Canada,27.0,1559.15,1802.0


In [56]:
simulacion_df.shape

(48, 4)

In [58]:
simulacion_df.columns

Index(['team', 'rank', 'total_points', 'elo'], dtype='str')

In [ ]:
selecciones.columns

In [57]:
simulacion_df.to_csv(
    "selecciones_mundial_2026.csv",
    index=False
)
